# 20.3 音频与语音 / Audio & Speech Processing

**中文**:声音是数据科学里一个自成一派的领域——语音识别、说话人识别、音乐分类、环境声检测、声学异常(设备故障)。音频看似只是一串数字(波形),但**直接对波形做机器学习效果很差**。关键洞察:**声音的信息藏在频率里**,不在时域波形里。本节从零走通音频 ML 的经典流水线:**波形 → FFT → 频谱图(spectrogram)→ 梅尔谱(mel)→ MFCC → CNN 分类**。这套"把音频变成图像再用 CNN"的思路,是深度学习前音频领域的黄金标准,至今仍是基础。
**English**: Sound is a field of its own in data science — speech recognition, speaker ID, music classification, environmental-sound detection, acoustic anomaly (equipment failure). Audio looks like just a string of numbers (a waveform), but **doing ML directly on the waveform works poorly**. The key insight: **a sound's information lives in its frequencies**, not the time-domain waveform. This section walks the classic audio-ML pipeline from scratch: **waveform → FFT → spectrogram → mel → MFCC → CNN classification**. This "turn audio into an image, then use a CNN" idea was the gold standard before deep learning and remains foundational.

---

**中文**:音频的表示层层递进:
**English**: Audio representations, step by step:
- **① 波形(waveform)**:声音是气压随时间的波动,采样成一串数字(采样率=每秒采多少个点,如 8000Hz)。直接看波形,人和机器都很难分辨"这是什么声音"。
  **Waveform**: sound is air-pressure fluctuation over time, sampled into numbers (sampling rate = points per second, e.g. 8000Hz). From the raw waveform, neither humans nor machines can easily tell "what sound is this."
- **② FFT(快速傅里叶变换)**:把一段信号从**时域**变到**频域**——告诉你这段声音里含有哪些频率、各有多强。这是理解声音的钥匙(一个 300Hz 的纯音,FFT 里就是 300Hz 处一根尖峰)。
  **FFT**: transforms a signal from the **time domain** to the **frequency domain** — telling you which frequencies are present and how strong. The key to understanding sound (a pure 300Hz tone is a single spike at 300Hz in the FFT).
- **③ 频谱图(spectrogram)= STFT**:声音的频率是**随时间变化**的(一句话里不同音节频率不同)。**短时傅里叶变换(STFT)** 把信号切成小窗、每窗做 FFT,得到一张**"时间×频率"的二维图**——横轴时间、纵轴频率、颜色是强度。这就把音频变成了图像。
  **Spectrogram = STFT**: a sound's frequencies **change over time** (different syllables in a sentence have different frequencies). The **Short-Time Fourier Transform (STFT)** slices the signal into small windows, FFTs each, giving a 2-D **"time × frequency" image** — time on x, frequency on y, color = intensity. This turns audio into an image.
- **④ 梅尔谱(mel spectrogram)**:人耳对低频敏感、对高频粗糙(1000Hz 和 1100Hz 能分清,10000 和 10100 分不清)。**梅尔刻度**按人耳感知重新分配频率轴,用一组三角滤波器把线性频率压成梅尔频率。更贴近人的听觉。
  **Mel spectrogram**: human ears are sensitive to low frequencies and coarse at high (we distinguish 1000 vs 1100Hz but not 10000 vs 10100Hz). The **mel scale** redistributes the frequency axis by human perception, using triangular filters to compress linear frequency into mel frequency — closer to human hearing.
- **⑤ MFCC(梅尔频率倒谱系数)**:对 log 梅尔谱再做一次 DCT(离散余弦变换),提取最紧凑的十几个系数。**深度学习前语音识别的标准特征**(GMM-HMM 时代的绝对主力)。
  **MFCC (Mel-Frequency Cepstral Coefficients)**: apply a DCT to the log-mel spectrogram, extracting the most compact ~13 coefficients. **The standard feature for speech recognition before deep learning** (the workhorse of the GMM-HMM era).

> 💡 **面试速查 / Interview cheat-sheet（★★ 音频/语音必考）**
> **中文**:音频信息在**频率**不在波形。流水线:波形→**FFT**(时域→频域)→**STFT 频谱图**(切窗做FFT, 得到时间×频率二维图)→**梅尔谱**(按人耳感知压频率轴, 三角滤波器)→**MFCC**(log梅尔+DCT, 传统语音标准特征)。**关键思路:把音频变成图像→用 CNN**(音频分类=图像分类)。深度学习时代:①端到端(直接学特征, 如 wav2vec2.0 自监督)、②**Whisper**(Transformer 语音识别)、③梅尔谱仍是主流输入。参数:窗长(时频分辨率权衡, 长窗频率准/时间糊)、hop(帧移)、n_mels。用途:语音识别(ASR)、说话人识别、声音事件检测、声学故障诊断、音乐信息检索。
> **English**: Audio information is in **frequencies**, not the waveform. Pipeline: waveform → **FFT** (time→frequency) → **STFT spectrogram** (windowed FFTs → a time×frequency image) → **mel spectrogram** (compress the frequency axis by perception, triangular filters) → **MFCC** (log-mel + DCT, the classic speech feature). **Key idea: turn audio into an image → use a CNN** (audio classification = image classification). In deep learning: ① end-to-end (learn features directly, e.g. wav2vec 2.0 self-supervised), ② **Whisper** (Transformer speech recognition), ③ mel spectrograms are still the mainstream input. Params: window length (time-frequency resolution trade-off — long window = sharp frequency, blurry time), hop, n_mels. Uses: speech recognition (ASR), speaker ID, sound-event detection, acoustic fault diagnosis, music information retrieval.


In [ ]:

# ============================================================
# 合成 4 类声音 / synthesize 4 sound classes
# 中文:低音、高音、上升啁啾(chirp)、双音。每类频率结构不同, 但直接看波形几乎分不出。
# English: low tone, high tone, rising chirp, two-tone. Different frequency structures, indistinguishable in the waveform.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy.fftpack import dct
rng=np.random.default_rng(0)
sr=8000; dur=0.5; n=int(sr*dur); t=np.linspace(0,dur,n,endpoint=False)   # 采样率/时长/时间轴
CLASS_NAMES=["低音 low","高音 high","上升啁啾 chirp","双音 two-tone"]
def make_sound(cls):
    if   cls==0: s=np.sin(2*np.pi*300*t)                                  # 300Hz 纯音 / low tone
    elif cls==1: s=np.sin(2*np.pi*1800*t)                                 # 1800Hz 纯音 / high tone
    elif cls==2: s=np.sin(2*np.pi*(300+2600*t/dur)*t)                     # 300→2900Hz 啁啾 / rising chirp
    else:        s=np.sin(2*np.pi*500*t)+np.sin(2*np.pi*1500*t)           # 500+1500Hz 双音 / two-tone
    return s + rng.normal(0,0.1,n)                                        # 加噪声 / add noise
print(f"采样率 {sr}Hz, 每段 {n} 个采样点 / {n} samples per {dur}s clip, {len(CLASS_NAMES)} 类")


**中文**:**① FFT:时域 → 频域**。对一段声音做 FFT,看它含哪些频率。低音在 300Hz 有尖峰,高音在 1800Hz,双音有两个峰——**频域一眼看穿声音的构成**,而时域波形看不出。
**English**: **① FFT: time → frequency**. FFT a sound to see its frequencies. The low tone spikes at 300Hz, the high tone at 1800Hz, the two-tone has two peaks — **the frequency domain reveals a sound's makeup at a glance**, which the waveform can't.


In [ ]:

# ============================================================
# ① FFT / Fast Fourier Transform
# ============================================================
def fft_spectrum(sig):
    spectrum=np.abs(np.fft.rfft(sig))                        # 幅度谱 / magnitude spectrum
    freqs=np.fft.rfftfreq(len(sig), 1/sr)                    # 对应频率 / frequencies
    return freqs, spectrum
for cls in range(4):
    f,sp=fft_spectrum(make_sound(cls)); peak=f[np.argmax(sp)]
    print(f"{CLASS_NAMES[cls]:<16} 主频峰值 / dominant frequency: {peak:.0f} Hz")


**中文**:**② STFT 频谱图**。啁啾声的频率**随时间上升**——单次 FFT 只给一个"平均频谱",看不到这种变化。STFT 把信号切成小窗、每窗 FFT,得到时间×频率的二维图,啁啾会显示成一条**斜线**。这就是把音频变成图像的关键。
**English**: **② STFT spectrogram**. The chirp's frequency **rises over time** — a single FFT gives one "average spectrum" and misses this. STFT slices into windows and FFTs each, yielding a time×frequency image where the chirp appears as a **diagonal line**. This is the key to turning audio into an image.


In [ ]:

# ============================================================
# ② STFT 频谱图 + ③ 梅尔滤波器组 + ④ MFCC / STFT + mel filterbank + MFCC
# ============================================================
def stft(sig, win=256, hop=128):
    w=np.hanning(win); frames=[]
    for start in range(0, len(sig)-win, hop):
        frames.append(np.abs(np.fft.rfft(sig[start:start+win]*w)))       # 每窗 FFT(加汉宁窗)/ windowed FFT
    return np.array(frames).T                                            # (频率, 时间) / (freq, time)

def mel_filterbank(n_filters, n_fft, sr):                                # 梅尔三角滤波器组 / mel filterbank
    hz2mel=lambda h: 2595*np.log10(1+h/700); mel2hz=lambda m: 700*(10**(m/2595)-1)
    mel_pts=np.linspace(hz2mel(0), hz2mel(sr/2), n_filters+2)
    bins=np.floor((n_fft+1)*mel2hz(mel_pts)/sr).astype(int)
    fb=np.zeros((n_filters, n_fft//2+1))
    for i in range(1, n_filters+1):
        for j in range(bins[i-1],bins[i]): fb[i-1,j]=(j-bins[i-1])/(bins[i]-bins[i-1])
        for j in range(bins[i],bins[i+1]): fb[i-1,j]=(bins[i+1]-j)/(bins[i+1]-bins[i])
    return fb
FB=mel_filterbank(40, 256, sr)

def audio_features(sig):
    S=stft(sig)                                                          # 频谱图 / spectrogram
    mel=FB@S                                                             # 梅尔谱 / mel spectrogram
    log_mel=np.log(mel+1e-6)                                             # 取对数(贴近感知)/ log mel
    mfcc=dct(log_mel, axis=0, norm="ortho")[:13]                        # MFCC = log梅尔的DCT前13维 / MFCC
    return S, log_mel, mfcc
S,logmel,mfcc=audio_features(make_sound(2))
print(f"频谱图 spectrogram: {S.shape} (频率×时间), 梅尔谱 log-mel: {logmel.shape}, MFCC: {mfcc.shape}")
print("梅尔谱把 129 个线性频率压成 40 个感知频率; MFCC 再压成 13 维紧凑特征")


**中文**:**⑤ CNN 分类:把音频当图像**。既然梅尔谱是二维"图像",就能直接用 CNN(Part 10)分类。我们对每段声音算梅尔谱,喂给一个小 CNN,看它能否分辨四类声音。
**English**: **⑤ CNN classification: audio as an image**. Since the mel spectrogram is a 2-D "image," we can classify it with a CNN (Part 10). We compute the mel spectrogram for each sound and feed a small CNN to distinguish the four classes.


In [ ]:

# ============================================================
# CNN 在梅尔谱上分类 / CNN classification on mel spectrograms
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F, time
torch.manual_seed(0); rng=np.random.default_rng(1)
X=[]; y=[]
for _ in range(800):
    c=rng.integers(4); _,logmel,_=audio_features(make_sound(c)); X.append(logmel); y.append(c)
X=torch.tensor(np.array(X),dtype=torch.float32).unsqueeze(1); y=torch.tensor(y)
X=(X-X.mean())/X.std()                                       # 标准化 / normalize
tr=slice(0,640); te=slice(640,800)
class AudioCNN(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(1,16,3,padding=1); s.c2=nn.Conv2d(16,32,3,padding=1); s.fc=nn.Linear(32,4)
    def forward(s,x):
        x=F.max_pool2d(F.relu(s.c1(x)),2); x=F.max_pool2d(F.relu(s.c2(x)),2)
        return s.fc(x.mean(dim=(2,3)))                        # 全局平均池化 / global average pool
m=AudioCNN(); opt=torch.optim.Adam(m.parameters(),3e-3)
t0=time.time()
for e in range(60):
    perm=torch.randperm(640)
    for b in range(0,640,64):
        idx=perm[b:b+64]; opt.zero_grad(); F.cross_entropy(m(X[tr][idx]),y[tr][idx]).backward(); opt.step()
acc=(m(X[te]).argmax(1)==y[te]).float().mean().item()
print(f"CNN 在梅尔谱上的分类准确率 / accuracy: {acc:.2f}  ({time.time()-t0:.0f}s)")
print("→ 把音频变成图像后, 图像 CNN 直接搞定音频分类 / audio-as-image + CNN solves it")


In [ ]:

# ============================================================
# 可视化:四类声音的波形/频谱/梅尔谱/MFCC / visualize the pipeline
# ============================================================
fig,ax=plt.subplots(2,4,figsize=(17,7))
for cls in range(4):
    sig=make_sound(cls); S,logmel,mfcc=audio_features(sig)
    ax[0,cls].plot(t[:800],sig[:800],lw=0.5,color="#4C72B0")   # 波形(看不出区别)/ waveform
    ax[0,cls].set_title(f"{CLASS_NAMES[cls]}\n波形(难分辨)"); ax[0,cls].set_xticks([]); ax[0,cls].set_yticks([])
    ax[1,cls].imshow(logmel,origin="lower",aspect="auto",cmap="magma")   # 梅尔谱(一目了然)/ mel spectrogram
    ax[1,cls].set_title("梅尔谱(清晰可分)"); ax[1,cls].set_xlabel("时间"); ax[1,cls].set_ylabel("梅尔频率" if cls==0 else "")
plt.tight_layout(); plt.savefig("/tmp/adv03_viz.png",dpi=80); plt.show()
print("上排波形几乎一样, 下排梅尔谱一眼区分:低音(底部亮)/高音(顶部亮)/啁啾(斜线)/双音(两条横线)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **音频的信息在频域,不在时域**:看可视化上排——四类声音的**波形几乎长得一样**,人和模型都难分辨。但下排的**梅尔谱一眼可分**:低音底部亮、高音顶部亮、啁啾是斜线、双音是两条横线。**把音频从时域搬到时频域,是音频 ML 的第一原理**。直接对原始波形做全连接网络效果差,正是因为信息被"藏"在了频率结构里。
2. **"音频变图像 + CNN"是简单而强大的范式**:一旦有了梅尔谱这张二维图,音频分类就**退化成图像分类**——直接套用 Part 10 的 CNN,本例轻松达到 100% 准确率。这套流水线(mel spectrogram → CNN)至今仍是音频分类、声音事件检测、声学异常诊断的**主力基线**,简单可靠。
3. **诚实的演进与现实**:①**MFCC 是深度学习前的王者**(GMM-HMM 语音识别),现在端到端深度模型常直接吃梅尔谱、甚至原始波形(wav2vec2.0 自监督学特征、Whisper 用 Transformer 做 ASR),MFCC 的重要性下降但仍是理解音频的基础;②**窗长是关键权衡**——长窗频率分辨率高但时间模糊(测不准原理的体现),要按任务调;③真实音频问题远比本例难(背景噪声、混响、多说话人、口音),需要数据增强(加噪、变速、SpecAugment)和大数据;④别忘了音频也可以做**自监督/迁移学习**(用大规模无标注音频预训练)。

**English**:
1. **Audio information is in the frequency domain, not the time domain**: see the top row of the visualization — the four classes' **waveforms look nearly identical**, hard for humans or models to tell apart. But the bottom-row **mel spectrograms are instantly separable**: the low tone is bright at the bottom, the high tone at the top, the chirp is a diagonal, the two-tone is two horizontal lines. **Moving audio from the time domain to the time-frequency domain is the first principle of audio ML.** A fully-connected net on the raw waveform works poorly precisely because information is "hidden" in the frequency structure.
2. **"Audio as image + CNN" is a simple, powerful paradigm**: once you have the 2-D mel spectrogram, audio classification **reduces to image classification** — apply the Part 10 CNN directly, reaching 100% accuracy here. This pipeline (mel spectrogram → CNN) is still the workhorse baseline for audio classification, sound-event detection, and acoustic anomaly diagnosis — simple and reliable.
3. **Honest evolution and reality**: ① **MFCC was king before deep learning** (GMM-HMM speech recognition); modern end-to-end models often ingest the mel spectrogram or even the raw waveform (wav2vec 2.0 self-supervised feature learning, Whisper's Transformer ASR), so MFCC's importance has waned but it remains foundational; ② **window length is a key trade-off** — a long window gives sharp frequency but blurry time (the uncertainty principle), tuned per task; ③ real audio problems are far harder (background noise, reverberation, multiple speakers, accents), needing augmentation (noise, time-stretch, SpecAugment) and big data; ④ audio also does **self-supervised / transfer learning** (pretrain on massive unlabeled audio).

> 💼 **实战视角 / Practical angle**
> **中文**:音频 ML 的实战栈:预处理用 **`librosa`**(梅尔谱/MFCC/重采样一行搞定)或 `torchaudio`;模型从 **CNN(梅尔谱)** 起步, 复杂任务用 **Whisper(ASR)、wav2vec2.0(自监督)、CLAP(音频-文本对比)**;数据增强用 **SpecAugment**(遮频遮时)。典型任务:①**语音识别 ASR**(现在直接用 Whisper);②**声音事件检测**(安防、工业监测);③**声学异常/故障诊断**(设备振动/噪声, 结合 Part 14 时序 + 20.4 异常检测);④**说话人识别/声纹**;⑤音乐推荐/标签。面试金句:*"音频信息在频域: 波形→FFT→STFT频谱图→梅尔谱(按人耳感知)→MFCC; 核心范式是'把音频变成图像用 CNN'; 现代用 Whisper/wav2vec 端到端, 但梅尔谱仍是主流输入。"*
> **English**: The audio-ML stack: preprocess with **`librosa`** (mel/MFCC/resample in one line) or `torchaudio`; models start with a **CNN on mel spectrograms**, complex tasks use **Whisper (ASR), wav2vec 2.0 (self-supervised), CLAP (audio-text contrastive)**; augment with **SpecAugment** (mask frequency/time). Typical tasks: ① **ASR** (now often just use Whisper); ② **sound-event detection** (security, industrial monitoring); ③ **acoustic anomaly/fault diagnosis** (equipment vibration/noise, combining Part 14 time series + 20.4 anomaly detection); ④ **speaker ID / voiceprint**; ⑤ music recommendation/tagging. Interview line: *"Audio information is in the frequency domain: waveform→FFT→STFT spectrogram→mel spectrogram (perceptual)→MFCC; the core paradigm is 'turn audio into an image and use a CNN'; modern systems use Whisper/wav2vec end-to-end, but the mel spectrogram is still the mainstream input."*

---
### 小结 / Summary
- **中文**:音频信息在频域; 流水线:波形→FFT→STFT频谱图→梅尔谱(人耳感知)→MFCC(log梅尔+DCT)。
- **English**: Audio information is in the frequency domain; pipeline: waveform→FFT→STFT spectrogram→mel spectrogram (perceptual)→MFCC (log-mel + DCT).
- **中文**:核心范式=把音频变成图像→用 CNN 分类(本例梅尔谱+CNN 达 100%)。
- **English**: Core paradigm = turn audio into an image → classify with a CNN (mel spectrogram + CNN hits 100% here).
- **中文**:MFCC 是传统语音标准特征; 现代用 Whisper/wav2vec 端到端, 但梅尔谱仍是主流输入。
- **English**: MFCC is the classic speech feature; modern systems use Whisper/wav2vec end-to-end, but the mel spectrogram is still the mainstream input.
